In [ ]:
#Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Importar las librerias
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBRegressor
import joblib
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

#Cargar el dataset para entrenar el modelo y el dataset para realizar
#una proyeccion de un trimestre
ruta_2024 = '/content/drive/My Drive/Anexos-CD/Datos/dataset_2024_sin_nombres.csv' #data set completo
ruta_2025 = '/content/drive/My Drive/Anexos-CD/Datos/dataset_2025_sin_nombres.csv' #data set con 2 trimestres

df24 = pd.read_csv(ruta_2024)
df25 = pd.read_csv(ruta_2025)

In [ ]:
#Prueba de proyectar el 3 trimestre de la gestion 2025
# ================================
#  PROYECCIÓN DEL 3T PARA 2025
# ================================

# Filtrar solo 2T del 2025
df25_2T = df25[df25['trimestre_num' ] == 2 ].copy()
df25Ini = pd.read_csv(ruta_2025)

# Crear copia para proyectar 3T
df25_3T = df25_2T.copy()
df25_3T = df25_3T.reset_index(drop=True)
df25_3T["trimestre_num"] = 3

# Eliminar columna "nota" porque la vamos a predecir y "trimestre" porque ya hay la columna
if "nota" in df25_3T.columns:
    df25_3T = df25_3T.drop(columns=["nota","trimestre"])

# ================================
#  ONE-HOT ENCODING
# ================================

encoder = joblib.load("/content/drive/My Drive/Anexos-CD/Encoders/encoder_xgboost.joblib")
xgboost = joblib.load("/content/drive/My Drive/Anexos-CD/Modelos/modelo_xgboost_notas.joblib")

# Aplicar transform al subset categórico
df25_3T_encoded = encoder.transform(df25_3T[cat_cols])

# Crear DataFrame con nombres de columnas del encoder
df25_3T_encoded = pd.DataFrame(df25_3T_encoded,
                               columns=encoder.get_feature_names_out(cat_cols),
                               index=df25_3T.index)
# Ordenamos la columnas numericas
cols_nums = ["trimestre_num","matricula"]
# Juntar con las columnas numéricas
df25_3T_final = pd.concat([
    df25_3T[cols_nums].reset_index(drop=True),
    df25_3T_encoded.reset_index(drop=True)
], axis=1)

# Verificar que NO quede ninguna columna object
print(df25_3T_final.dtypes)

# Convertimos a Numpy Array
N_pred = df25_3T_final.to_numpy()

# ================================
#  PREDICCIÓN CON XGBOOST
# ================================

df25_3T_final["nota_pred"] = xgboost.predict(N_pred)

df25_3T_final

# ============================
# 🔸 GUARDAR CSV FINAL
# ============================
df25_3T["nota"] = df25_3T_final["nota_pred"]
df25_3T["trimestre"] = "3T"
df25_3T["nota"] = df25_3T["nota"].fillna(0).astype(int)
df25_completo_final = pd.concat([df25Ini, df25_3T], ignore_index=True)
df25_completo_final = df25_completo_final.drop(columns= ["trimestre_num"])
df25_completo_final.to_csv("/content/drive/MyDrive/Anexos-CD/Predicciones/XGBoost/proyeccion_3T_2025_XGBoost.csv", index=False)


print("✔ Archivo generado: proyeccion_3T_2025_XGBoost.csv")
df25_3T_final.head()

In [ ]:
from scipy.stats import gaussian_kde
ruta_20253 = '/content/drive/My Drive/Anexos-CD/Predicciones/XGBoost/proyeccion_3T_2025_XGBoost.csv' #data set final
ruta_20250 = '/content/drive/My Drive/Anexos-CD/Datos/dataset_2025_sin_nombres.csv' #data set con 2 trimestres

df3t = pd.read_csv(ruta_20253)
df253 = pd.read_csv(ruta_20250)


# -----------------------------
# Separar los datos necesarios
# -----------------------------
df_1T = df253[df253["trimestre"] == "1T"].copy()
df_2T = df253[df253["trimestre"] == "2T"].copy()

# Proyecciones del modelo (3T)
df_3T_pred = df3t[df3t["trimestre"] == "3T"].copy()

#plt.figure(figsize=(10, 6))

# KDE de 1T
kde_1T = gaussian_kde(df_1T["nota"])
x_vals = np.linspace(0, 100, 500)
plt.plot(x_vals, kde_1T(x_vals), label="1T Real")

# KDE de 2T
kde_2T = gaussian_kde(df_2T["nota"])
x_vals = np.linspace(0, 100, 500)
plt.plot(x_vals, kde_2T(x_vals), label="2T Real")

# KDE de 3T
kde_3T = gaussian_kde(df_3T_pred["nota"])
plt.plot(x_vals, kde_3T(x_vals), label="3T Proyectado")

plt.title("Comparación de distribuciones KDE: 1T, 2T Real vs 3T Proyectado XGB")
plt.xlabel("Notas")
plt.ylabel("Densidad")
plt.legend()
plt.grid(True)
plt.show()

# ============================
# BOXPLOT
# ============================

plt.figure(figsize=(8, 5))

plt.boxplot(
    [df_1T["nota"], df_2T["nota"], df_3T_pred["nota"]],
    labels=["1T Real","2T Real", "3T Proyectado"]
)

plt.title("Boxplot comparativo: 1T Real vs 2T Real vs 3T Proyectado XGB")
plt.ylabel("Notas")
plt.grid(True)
plt.show()

In [ ]:
from scipy.stats import spearmanr, ks_2samp
# Comparacion de Metricas
# EMPAREJAR 2T REAL VS 3T PROYECTADO

df_eval = pd.merge(
    df_2T,
    df_3T_pred,
    on=["matricula", "curso", "materia", "nivel"],
    suffixes=("_2T", "_3T"),
    how="inner"
)

print("Filas emparejadas para evaluación:", len(df_eval))

# MAE y RMSE

mae = mean_absolute_error(df_eval["nota_2T"], df_eval["nota_3T"])
rmse = np.sqrt(mean_squared_error(df_eval["nota_2T"], df_eval["nota_3T"]))
r2 = r2_score(df_eval["nota_2T"], df_eval["nota_3T"])
print("MAE emparejado:", mae)
print("RMSE emparejado:", rmse)
print("R2 emparejado:", r2)